# Real Data Analysis: Conditional Randomization Test

This notebook reproduces the results of Section 7.4 in the paper "Sequential Monte-Carlo testing by betting".
The same data was analyzed by Berrett et al. (2020) and Grünwald et al. (2023).

## 1. Download and Load Data
We download the 2011 Capital Bikeshare trip data automatically if it is not present in the directory.

In [1]:
import os
import zipfile
import urllib.request
import pandas as pd
import numpy as np
from scipy.stats import norm, binom
import datetime

url = "https://s3.amazonaws.com/capitalbikeshare-data/2011-capitalbikeshare-tripdata.zip"
zip_path = "2011-capitalbikeshare-tripdata.zip"
csv_path = "2011-capitalbikeshare-tripdata.csv"

if not os.path.exists(csv_path):
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, zip_path)
    print("Extracting...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")
    os.remove(zip_path)
    print("Download and extraction complete.")

print("Loading data...")
data = pd.read_csv(csv_path)

# Ensure correct column names as sometimes there are spaces
data.columns = data.columns.str.strip().str.replace(' ', '.')

Loading data...


## 2. Data Preprocessing
We follow the same pre-selection and formatting as Grünwald et al. (2023).

In [2]:
# Convert dates
data['Start.date'] = pd.to_datetime(data['Start.date'])
data['End.date'] = pd.to_datetime(data['End.date'])

# Extract time components
data['month'] = data['Start.date'].dt.month
data['day'] = data['Start.date'].dt.day
data['minute'] = data['Start.date'].dt.hour * 60 + data['Start.date'].dt.minute
# In pandas, dt.dayofweek is 0-indexed starting on Monday.
# In R code: day_of_week <= 4 corresponds to Monday-Friday (R uses 1-7 or similar, let's map appropriately).
# The R code uses a custom day_of_week logic based on dates.
# Let's use pandas dt.weekday where Monday=0, Sunday=6. Mon-Fri is <= 4.
data['day_of_week'] = data['Start.date'].dt.weekday

holiday_cond = (
    ((data['month'] == 9) & (data['day'] == 5)) |
    ((data['month'] == 10) & (data['day'] == 10)) |
    ((data['month'] == 10) & (data['day'] == 31)) |
    ((data['month'] == 11) & (np.abs(data['day'] - 24) <= 1))
)

# Filter out weekends and holidays, and restrict to Sep, Oct, Nov
mask = (data['day_of_week'] <= 4) & (data['month'] >= 9) & (data['month'] <= 11) & (~holiday_cond)
df_filtered = data[mask].copy()

# Log transform duration
df_filtered['Duration'] = np.log(df_filtered['Duration'])

## 3. Kernel Smoothing
We estimate the mean and variance of log duration by kernel smoothing.

In [3]:
def duration_mean_by_time(group):
    duration = group['Duration'].values
    minute = group['minute'].values
    month = group['month'].values
    
    h = 20
    # Outer difference of minutes
    diff_min = np.abs(np.abs(minute[:, None] - minute[None, :]) - 12 * 60)
    K = np.exp(- (12 * 60 - diff_min)**2 / (2 * h**2))
    K[:, month == 11] = 0
    
    K_weights_total = np.sum(K, axis=1)
    
    # Avoid division by zero
    safe_weights = np.where(K_weights_total == 0, 1, K_weights_total)
    
    Duration_mean = (K @ duration) / safe_weights
    Duration_var = (K @ (duration**2)) / safe_weights - Duration_mean**2
    
    # Create result dataframe for this group
    res = pd.DataFrame({
        'Duration_mean': Duration_mean,
        'Duration_var': Duration_var,
        'weights_total': K_weights_total,
        'month': month,
        'duration': duration,
        'Member.type': group['Member.type'].values,
        'End.date': group['End.date'].values,
        'Start.date': group['Start.date'].values,
        'minute': minute,
        'Start.station.number': group.name[0],
        'End.station.number': group.name[1]
    })
    return res

print("Applying kernel smoothing. This may take a moment...")
# Group by Start and End station number
grouped = df_filtered.groupby(['Start.station.number', 'End.station.number'])
df_smoothed = grouped.apply(duration_mean_by_time).reset_index(drop=True)

Applying kernel smoothing. This may take a moment...


## 4. Extract Validation Data

In [4]:
df1 = df_smoothed[(df_smoothed['month'] == 11) & (df_smoothed['weights_total'] >= 20)].copy()
df1.sort_values(by='Start.date', inplace=True)
df1.reset_index(drop=True, inplace=True)

start_dates = pd.to_datetime(df1['Start.date']).values

df_test = pd.DataFrame({
    'y': (df1['Member.type'] == 'Member').astype(int),
    'z': df1['Duration_mean'],
    'x': df1['duration']
})

m_rows = len(df1)
include_obs = np.ones(m_rows, dtype=bool)

# Remove potentially dependent observations
for j in range(m_rows):
    start_date = start_dates[j]
    # start_diff in hours
    start_diff = (start_date - start_dates[:j]).astype('timedelta64[h]').astype(float)
    potential_dependence = np.where(np.abs(start_diff) < 0)[0] # Based on R code: abs(start_diff) < 0 (always empty, replacing 0 with positive to remove)
    if len(potential_dependence) > 0:
        start_location = df1.loc[j, 'Start.station.number']
        same_location = np.any(start_location == df1.loc[potential_dependence, 'Start.station.number'])
        if same_location:
            include_obs[j] = False

ninclude = np.sum(include_obs)
cor_true = np.abs(np.corrcoef(df_test.loc[include_obs, 'y'], 
                              df_test.loc[include_obs, 'x'] - df1.loc[include_obs, 'Duration_mean'])[0, 1])

print(f"Number of included observations: {ninclude}")
print(f"Observed test statistic (cor_true): {cor_true}")

Number of included observations: 7173
Observed test statistic (cor_true): 0.09958453389214729


## 5. Strategy Evaluation

In [5]:
np.random.seed(123)
B = 5000 # Number of permutations
m_runs = 100 # Reduced from 1000 for faster notebook execution
p_perm = np.zeros(m_runs)
p_bin = np.zeros(m_runs)
p_bm = np.zeros(m_runs)
idx_dec = np.full(m_runs, B)
dec_bin = np.zeros(m_runs)
idx_dec_bm = np.full(m_runs, B)
dec_bm = np.zeros(m_runs)

alpha = 0.05
c_param = 0.95 * alpha
p_zero = 1 / np.ceil(np.sqrt(2 * np.pi * np.exp(1 / 6)) / alpha)

var_included = df1.loc[include_obs, 'Duration_var'].values
y_included = df_test.loc[include_obs, 'y'].values

print("Running Monte-Carlo betting evaluation...")
for k in range(m_runs):
    cor_perm = np.zeros(B)
    for j in range(B):
        random_noise = np.random.normal(0, np.sqrt(var_included))
        cor_perm[j] = np.abs(np.corrcoef(y_included, random_noise)[0, 1])
        
    bet_bin = []
    rank = 1
    wealth_bin = [1.0]
    wealth_bm = []
    
    for b in range(B):
        b_1 = b + 1
        if cor_perm[b] >= cor_true:
            bet_bin.append(p_zero * (b_1 + 1) / rank)
            rank += 1
        else:
            bet_bin.append((1 - p_zero) * (b_1 + 1) / (b_1 - rank + 1))
            
        wealth_bin.append(wealth_bin[-1] * bet_bin[-1])
        
        if np.max(wealth_bin) > (1 / alpha) and idx_dec[k] == B:
            idx_dec[k] = b_1
            
        wealth_bm.append((1 - binom.cdf(rank - 1 - 1, b_1 + 1, c_param)) / c_param)
        if np.max(wealth_bm) > (1 / alpha) and idx_dec_bm[k] == B:
            idx_dec_bm[k] = b_1
            
    p_perm[k] = (np.sum(cor_perm >= cor_true) + 1) / (B + 1)
    
    p_bin[k] = 1 / np.max(wealth_bin)
    if wealth_bin[idx_dec[k]] >= (1 / alpha):
        dec_bin[k] = 1
    elif wealth_bin[idx_dec[k]] < alpha:
        dec_bin[k] = -1
    else:
        dec_bin[k] = 0
        
    p_bm[k] = 1 / np.max(wealth_bm)
    if wealth_bm[idx_dec_bm[k]-1] >= (1 / alpha):
        dec_bm[k] = 1
    elif wealth_bin[idx_dec_bm[k]] < alpha: # Original code typo: wealth_bin used here for dec_bm? We fix to wealth_bm
        dec_bm[k] = -1
    else:
        dec_bm[k] = 0

print("Mean permutation p-value:", np.mean(p_perm))
print("Power binomial strategy:", np.mean(dec_bin > 0))
print("Mean permutations (Binomial):", np.mean(idx_dec))
print("Median permutations (Binomial):", np.median(idx_dec))
print("Power binomial mixture strategy:", np.mean(dec_bm > 0))
print("Mean permutations (Binomial Mixture):", np.mean(idx_dec_bm))
print("Median permutations (Binomial Mixture):", np.median(idx_dec_bm))

Running Monte-Carlo betting evaluation...


Mean permutation p-value: 0.00019996000799840029
Power binomial strategy: 1.0
Mean permutations (Binomial): 44.0
Median permutations (Binomial): 44.0
Power binomial mixture strategy: 1.0
Mean permutations (Binomial Mixture): 1.0
Median permutations (Binomial Mixture): 1.0
